# NCKH V7.3 — Context-Aware Detection Verification (pilot)

**Mục tiêu:** Giữ nguyên V7.2 parser-fixed đã chạy trên 1.301 cặp. Chỉ kiểm tra lại bằng Qwen3-VL-4B những cặp CountGD++ gốc trả 0, V7.2 vẫn trả 0 nhưng cache tiling có nhiều candidate. **Không dùng ground truth để chọn ảnh hoặc tạo dự đoán.**

**Ý tưởng mới:** V7.2 kiểm tra crop sát bounding box (ít ngữ cảnh). V7.3 đưa Qwen hai hình: ảnh toàn cảnh và bảng crop **có ngữ cảnh rộng, bounding box đánh dấu đỏ**, hỏi vật thể bên trong box có đúng target không. Dùng quy tắc đã chốt `full-image YES` và tối thiểu **4/6** local YES, đầy đủ 6 nhãn; nếu không đạt, giữ V7.2. Một lần chạy thử trên 5 trường hợp, không huấn luyện, không chạy lại CountGD++.

**Giới hạn nghiên cứu:** Đây là thử nghiệm *exploratory* trên TEST đã tái sử dụng. Chỉ có thể kết luận hiệu năng trên bộ đã xem; sau khi chốt quy tắc, cần negative/positive held-out mới để kiểm chứng khả năng tổng quát hoá. Không làm tăng số đếm chỉ vì có nhiều box.

**Inputs:** Add Input file `V7_3_Inputs_Ready_Kaggle.zip` từ Research Kit (Kaggle có thể tự giải nén) và dataset ảnh FSC-147 `hihihi5`. GPU, Internet ON để tải Qwen.

In [ ]:
# CELL 1 — CONFIG (đóng băng trước khi xem các phép đo)
import sys, subprocess, json, re, os, time, io, zipfile, hashlib, traceback
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageOps

ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUTPUT = ROOT/'V7_3_context_results'
OUTPUT.mkdir(exist_ok=True, parents=True)
INPUT_ROOTS = [p for p in [Path('/kaggle/input'),Path('/mnt/data')] if p.exists()]
QWEN_MODEL_ID='Qwen/Qwen3-VL-4B-Instruct'
MIN_TILED_BOXES=20    # điều kiện kỹ thuật để kiểm tra box, KHÔNG dùng GT
N_SAMPLE=6            # 6 box đa vùng
MIN_LOCAL_YES=4       # yêu cầu 4/6 local verification (như V7.2)
CONTEXT_MULTIPLIER=6.0  # 6x bbox thay vì crop quá sát
PANEL_SIZE=336
SAVE_SHEETS=True
RUN_MAX=None           # None => xét toàn bộ các trường hợp eligible (hiện tại 5)
REQUIRE_ORIGINAL_FULL_YES=True  # fail-closed nếu Qwen full image gốc NO/UNCERTAIN

print('Output:',OUTPUT)

In [ ]:
# CELL 2 — LOAD CACHE (đọc trực tiếp file nếu Kaggle tự giải nén, hoặc đọc ZIP)
import zipfile
PREFIX='V7_3_Inputs_Ready_Kaggle'
needed=['v72_frozen_1301_predictions.csv','v72_tiles_checkpoint.csv',
        'v72_visual_gate_checkpoint.csv']
CACHEROOT=OUTPUT/'inputs'
CACHEROOT.mkdir(exist_ok=True,parents=True)

# Ưu tiên đường dẫn file thực bên Kaggle Input
for name in needed:
    hits=[]
    for root in INPUT_ROOTS:
        hits.extend(root.rglob(name))
    if hits:
        (CACHEROOT/name).write_bytes(sorted(hits,key=lambda x:str(x))[0].read_bytes())

# Nếu thiếu file, thử extract ZIP do Kaggle giữ nguyên
if any(not (CACHEROOT/name).exists() for name in needed):
    zip_hits=[]
    for root in INPUT_ROOTS:
        zip_hits += list(root.rglob(PREFIX+'.zip'))
    if not zip_hits:
        raise FileNotFoundError('Hãy Add Input V7_3_Inputs_Ready_Kaggle.zip (có trong Research Kit).')
    with zipfile.ZipFile(sorted(zip_hits,key=str)[0]) as z:
        for name in z.namelist():
            if name in needed or (name.startswith('tile_boxes/') and name.endswith('.json')):
                target=CACHEROOT/name
                target.parent.mkdir(exist_ok=True,parents=True)
                target.write_bytes(z.read(name))

# Nếu 3 CSV có nhưng JSON được Kaggle giải nén riêng, chỉ copy JSON, không đụng các file khác.
if not (CACHEROOT/'tile_boxes').exists() or not list((CACHEROOT/'tile_boxes').glob('*.json')):
    for root in INPUT_ROOTS:
        for f in root.rglob('POS__*.jpg.json'):
            (CACHEROOT/'tile_boxes').mkdir(exist_ok=True)
            (CACHEROOT/'tile_boxes'/f.name).write_bytes(f.read_bytes())

# Thử thêm đường dẫn ZIP chung chứa tile_boxes nếu Kaggle không giữ tên ZIP inputs
if not (CACHEROOT/'tile_boxes').exists() or not list((CACHEROOT/'tile_boxes').glob('*.json')):
    for root in INPUT_ROOTS:
        for path in root.rglob('*.zip'):
            try:
                with zipfile.ZipFile(path) as z:
                    for name in z.namelist():
                        if name.startswith('tile_boxes/') and name.endswith('.json'):
                            dest=CACHEROOT/name;dest.parent.mkdir(parents=True,exist_ok=True)
                            dest.write_bytes(z.read(name))
            except zipfile.BadZipFile:continue
            if (CACHEROOT/'tile_boxes').exists():break

for name in needed:
    assert (CACHEROOT/name).exists(), f'Thiếu {name}. Hãy Add Input ZIP đi kèm notebook.'
base=pd.read_csv(CACHEROOT/needed[0]); tiled=pd.read_csv(CACHEROOT/needed[1]);oldgate=pd.read_csv(CACHEROOT/needed[2])
assert len(base)==1301 and base.group_id.is_unique, 'Bộ gốc phải có 1301 cặp duy nhất'
assert len(tiled)==27 and tiled.group_id.is_unique
assert len(oldgate)==27 and oldgate.group_id.is_unique
assert {'group_id','v72_fixed_count','baseline_count','v7_count','pair_type','gt_count'}.issubset(base)

# Chỉ tham chiếu prediction + cache inference, KHÔNG đọc gt_count hoặc pair_type cho eligibility
eligible=(base[['group_id','image_name','target','baseline_count','v72_fixed_count']]
          .merge(tiled[['group_id','tile_count','status']],on='group_id',how='inner')
          .query('baseline_count == 0 and v72_fixed_count == 0 and tile_count >= @MIN_TILED_BOXES and status == "ok"')
          .sort_values('image_name').reset_index(drop=True))
if RUN_MAX is not None:eligible=eligible.head(RUN_MAX).copy()
assert len(eligible)>0, 'Không có trường hợp cần xác minh thêm.'
for gid in eligible.group_id:
    f=CACHEROOT/'tile_boxes'/(gid+'.json')
    assert f.exists(), f'Thiếu tile boxes: {f}'

# Tìm ảnh JPG (ảnh phải có thật, không được suy ra từ nhãn)
image_index={}
for root in INPUT_ROOTS:
    for f in root.rglob('*.jpg'):
        if f.name not in image_index:image_index[f.name]=f
missing=set(eligible.image_name)-set(image_index)
if missing:raise FileNotFoundError(f'Thiếu ảnh FSC-147: {sorted(missing)[:8]}')

full_log=oldgate.set_index('group_id')
print('V7.3 số trường hợp cần chạy Qwen:',len(eligible))
print(eligible[['image_name','target','tile_count','v72_fixed_count']].to_string(index=False))
print('GT chỉ được dùng SAU suy luận, ở CELL đánh giá.')

### Cơ chế kiểm chứng độc lập với Ground Truth

- Chọn bằng **kết quả dự đoán**: `baseline_count=0`, `V7.2_count=0`, `tile_count>=20`; không truy cập `gt_count` trong quá trình chọn và kiểm tra.
- Qwen nhìn thấy **đối tượng nằm trong khung đỏ** và phần ngữ cảnh xung quanh, không chỉ crop sát box.
- Yêu cầu 6/6 nhãn hợp lệ, ít nhất 4 nhãn YES và bằng chứng hiện diện toàn ảnh gốc YES. Nếu thiếu/không rõ thì fail-closed giữ V7.2.
- Có checkpoint mỗi ảnh; khi Kaggle ngắt phiên, upload ZIP/CSV checkpoint ở lần chạy kế tiếp.

In [ ]:
# CELL 3 — DEPENDENCIES + QWEN (không cài torch/torchvision lại)
import torch,gc
assert torch.cuda.is_available(), 'Hãy bật GPU trong Kaggle Session options.'
try:
    from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
    from qwen_vl_utils import process_vision_info
except Exception:
    subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade',
                    'transformers>=4.57.0','accelerate','qwen-vl-utils'],check=True)
    from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
    from qwen_vl_utils import process_vision_info

processor=AutoProcessor.from_pretrained(QWEN_MODEL_ID)
model=Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID,dtype=torch.float16,device_map='auto',low_cpu_mem_usage=True).eval()
print('Loaded Qwen:',QWEN_MODEL_ID,'GPU:',torch.cuda.get_device_name(0))

In [ ]:
# CELL 4 — CONTEXT PANELS & OUTPUT PARSER (unit tests)
from PIL import Image, ImageDraw, ImageOps

LINE_RX=re.compile(r'^(?:ID\s*)?(\d+)\s*[|:]\s*(YES|NO|UNCERTAIN)\s*$',re.I)
FULL_RX=re.compile(r'\b(?:YES|NO|UNCERTAIN)\b',re.I)

def parse_full(text):
    tags=FULL_RX.findall(str(text).upper())
    return tags[0] if len(tags)==1 else 'UNCERTAIN'

def parse_panel(text,n):
    out={}
    for line in str(text).strip().splitlines():
        match=LINE_RX.match(line.strip())
        if match:
            idx=int(match.group(1))
            if 0<=idx<n and idx not in out:out[idx]=match.group(2).upper()
    return out

def choose_diverse(boxes,n=N_SAMPLE):
    boxes=sorted(boxes,key=lambda x:-float(x.get('score',0)))
    chosen=[];used_cells=set()
    for x in boxes:
        cell=tuple(x.get('cell',[0,0]))
        if cell not in used_cells:
            chosen.append(x);used_cells.add(cell)
        if len(chosen)>=n:return chosen
    used_ids={id(x) for x in chosen}
    for x in boxes:
        if id(x) not in used_ids:
            chosen.append(x)
            if len(chosen)>=n:break
    return chosen

def local_context_box(im_size, bbox, mult=CONTEXT_MULTIPLIER):
    W,H=im_size
    x1,y1,x2,y2=map(float,bbox)
    cx=(x1+x2)/2;cy=(y1+y2)/2
    bw=max(1,x2-x1);bh=max(1,y2-y1)
    crop_w=max(mult*bw,40);crop_h=max(mult*bh,40)
    # Ít nhất một bề rộng nhỏ vừa đủ nhìn rõ đối tượng nhưng không phóng quá 50% ảnh.
    crop_w=min(W,max(crop_w,2*bw));crop_h=min(H,max(crop_h,2*bh))
    left=max(0,min(W-crop_w,cx-crop_w/2));top=max(0,min(H-crop_h,cy-crop_h/2))
    return tuple(map(int,[round(left),round(top),round(left+crop_w),round(top+crop_h)]))

def sheet_context(image, chosen):
    n=len(chosen); cols=3;rows=(n+cols-1)//cols; head=32
    sheet=Image.new('RGB',(cols*PANEL_SIZE,rows*(PANEL_SIZE+head)),'white')
    draw=ImageDraw.Draw(sheet)
    for i,one in enumerate(chosen):
        box=one['box'];rect=local_context_box(image.size,box)
        crop=image.crop(rect).convert('RGB')
        panel=ImageOps.contain(crop,(PANEL_SIZE-12,PANEL_SIZE-12))
        x0=(i%cols)*PANEL_SIZE;y0=(i//cols)*(PANEL_SIZE+head)
        ox=x0+(PANEL_SIZE-panel.width)//2;oy=y0+(PANEL_SIZE-panel.height)//2
        sheet.paste(panel,(ox,oy))
        # Convert global box to panel space (contain preserves aspect ratio)
        scale=panel.width/max(1,crop.width)
        bx=[ox+(float(box[j])-rect[0])*scale if j%2==0 else oy+(float(box[j])-rect[1])*scale for j in range(4)]
        draw.rectangle(bx,outline='red',width=4)
        draw.rectangle((x0,y0+PANEL_SIZE,x0+PANEL_SIZE-1,y0+PANEL_SIZE+head-1),fill='white',outline='black',width=1)
        draw.text((x0+8,y0+PANEL_SIZE+5),f'ID {i} (red rectangle)',fill='black')
    return sheet

assert parse_full('YES')=='YES'
assert parse_full('NO')=='NO'
assert parse_full('YES, NO')=='UNCERTAIN'
assert parse_panel('0|YES\n1|NO\n2|UNCERTAIN',3)=={0:'YES',1:'NO',2:'UNCERTAIN'}
assert local_context_box((500,400),[10,10,20,20])[2]>10
print('Context + parser self-tests PASS')

In [ ]:
# CELL 5 — QWEN CONTEXTUAL VERIFICATION
@torch.inference_mode()
def qwen_context_check(image,target,chosen):
    sheet=sheet_context(image,chosen)
    prompt=(
        f'Target category: "{target}".\n'
        'Image 1 is the full original photograph. Image 2 has 6 numbered zoomed-in panels, each with a RED rectangle.\n'
        'For each panel, classify ONLY the object INSIDE THE RED rectangle. The rest of the panel provides visual context.\n'
        'Is the boxed object truly an instance of the target category? Similar-looking or related categories are NOT target.\n'
        'Do NOT label the panel positive merely because the target exists elsewhere in the full image.\n'
        'If the boxed object is unclear or cut off, output UNCERTAIN.\n'
        'Reply with EXACTLY six lines, one for each panel ID from 0 through 5. '
        'On each line write the ID, a vertical bar, and YOUR verdict YES/NO/UNCERTAIN. No explanations.'
    )
    messages=[{'role':'user','content':[{'type':'image','image':image},
                                        {'type':'image','image':sheet},
                                        {'type':'text','text':prompt}]}]
    text=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    imgs,vids,kwargs=process_vision_info(messages,image_patch_size=16,
                                        return_video_kwargs=True,return_video_metadata=True)
    metas=None
    if vids is not None:
        vids,metas=zip(*vids);vids=list(vids);metas=list(metas)
    inputs=processor(text=[text],images=imgs,videos=vids,video_metadata=metas,
                     padding=True,return_tensors='pt',do_resize=False,**kwargs)
    inputs=inputs.to(next(model.parameters()).device)
    gen=model.generate(**inputs,max_new_tokens=128,do_sample=False)
    raw=processor.batch_decode([out[len(inp):] for inp,out in zip(inputs.input_ids,gen)],
                               skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]
    return raw,sheet

In [ ]:
# CELL 6 — VERIFY + CHECKPOINT (nếu restart, Add Input checkpoint ZIP và Run All lại)
checkpoint=OUTPUT/'v73_context_verification_checkpoint.csv'
if not checkpoint.exists():
    for root in INPUT_ROOTS:
        files=list(root.rglob(checkpoint.name))
        if files:
            checkpoint.write_bytes(sorted(files,key=str)[0].read_bytes())
            print('Đã khôi phục checkpoint:',checkpoint.name)
            break

previous=pd.read_csv(checkpoint) if checkpoint.exists() else pd.DataFrame()
if len(previous) and previous.group_id.duplicated().any():
    raise ValueError('Checkpoint có group_id trùng')
records=previous.to_dict('records') if len(previous) else []
done={r['group_id'] for r in records if r.get('status')=='ok'}

for ix,r in enumerate(eligible.itertuples(index=False),1):
    gid=str(r.group_id)
    if gid in done:
        print('[SKIP]',gid);continue
    t0=time.time()
    boxes=json.loads((CACHEROOT/'tile_boxes'/(gid+'.json')).read_text(encoding='utf8'))
    samples=choose_diverse(boxes,N_SAMPLE)
    full_raw=full_log.loc[gid,'raw_full']
    full_label=parse_full(full_raw)
    rec={'group_id':gid,'image_name':r.image_name,'target':r.target,
         'tile_count':int(r.tile_count),'old_full_label':full_label,
         'old_full_raw':str(full_raw),'status':'error','raw_context':'',
         'local_yes':0,'local_no':0,'local_uncertain':0,
         'complete':False,'accepted':False,'new_count':int(r.v72_fixed_count),
         'error':''}
    try:
        assert len(samples)==N_SAMPLE
        img=Image.open(image_index[r.image_name]).convert('RGB')
        raw,sheet=qwen_context_check(img,r.target,samples)
        parsed=parse_panel(raw,N_SAMPLE)
        rec['raw_context']=raw
        rec['local_yes']=sum(x=='YES' for x in parsed.values())
        rec['local_no']=sum(x=='NO' for x in parsed.values())
        rec['local_uncertain']=sum(x=='UNCERTAIN' for x in parsed.values())
        rec['complete']=(len(parsed)==N_SAMPLE)
        rec['accepted']=bool(rec['complete'] and rec['local_yes']>=MIN_LOCAL_YES
                             and (not REQUIRE_ORIGINAL_FULL_YES or full_label=='YES'))
        rec['new_count']=int(r.tile_count) if rec['accepted'] else int(r.v72_fixed_count)
        rec['status']='ok'
        if SAVE_SHEETS:
            sheet.save(OUTPUT/(gid+'_context_panel.jpg'),quality=85)
    except Exception as e:
        rec['error']=repr(e)
        traceback.print_exc()
    rec['sec']=round(time.time()-t0,1)
    records=[old for old in records if old.get('group_id')!=gid]+[rec]
    pd.DataFrame(records).to_csv(checkpoint,index=False)
    print(f"[{ix}/{len(eligible)}] {gid} full={rec['old_full_label']} context YES={rec['local_yes']}/{N_SAMPLE} -> {rec['new_count']} status={rec['status']}",flush=True)
    if rec['status']!='ok':
        raise RuntimeError('Không tiếp tục sau lỗi Qwen: '+gid+' '+rec['error'])
print('Done Qwen:',len(records))

In [ ]:
# CELL 7 — INFERENCE-LOCKED RESULTS, SAU ĐÓ MỚI TÍNH GT METRICS
verified=pd.read_csv(checkpoint)
assert len(verified)==len(eligible) and verified.group_id.is_unique
assert (verified.status=='ok').all()

# Không dùng GT để lựa chọn nhánh hay số dự đoán
pred=base[['group_id','image_name','target','pair_type','source_class','gt_count','baseline_count','v7_count','v72_fixed_count']].copy()
lookup=verified.set_index('group_id')['new_count'].to_dict()
pred['v73_count']=pred.apply(lambda r:int(lookup.get(r.group_id,r.v72_fixed_count)),axis=1)
pred['v72_count']=pred.v72_fixed_count.astype(int)
pred['changed']=pred.v73_count.ne(pred.v72_count)
pred['err_v7']=(pred.v7_count-pred.gt_count).abs()
pred['err_v72']=(pred.v72_count-pred.gt_count).abs()
pred['err_v73']=(pred.v73_count-pred.gt_count).abs()
pred['delta_err']=pred.err_v73-pred.err_v72
assert len(pred)==1301 and pred.group_id.is_unique

metrics=[]
for kind,sub in [('POSITIVE',pred[pred.pair_type=='POSITIVE']),
                 ('HARD_NEGATIVE',pred[pred.pair_type=='HARD_NEGATIVE']),('ALL',pred)]:
    for key,col in [('V7','v7_count'),('V7.2_FIXED','v72_count'),('V7.3_CONTEXT','v73_count')]:
        err=(sub[col]-sub.gt_count).abs()
        metrics.append(dict(subset=kind,method=key,n=len(sub),mae=float(err.mean()),
            rmse=float(np.sqrt((err**2).mean())),
            fpr=float((sub[col]>0).mean()) if kind=='HARD_NEGATIVE' else np.nan,
            positive_zero_rate=float((sub[col]==0).mean()) if kind=='POSITIVE' else np.nan))
met=pd.DataFrame(metrics)
print(met.to_string(index=False))
print('Changed:',int(pred.changed.sum()),'| rejected eligible:',int((~verified.accepted).sum()))
print('Change errors (+ worse):',pred.delta_err.sum())

pred.to_csv(OUTPUT/'v73_1301_predictions.csv',index=False)
met.to_csv(OUTPUT/'v73_1301_metrics.csv',index=False)
verified.to_csv(OUTPUT/'v73_verification_audit.csv',index=False)
pred[pred.changed].sort_values('delta_err').to_csv(OUTPUT/'v73_changed_cases.csv',index=False)

In [ ]:
# CELL 8 — EXPORT ZIP, CÓ THỂ ADD INPUT CHO LẦN TIẾP
from IPython.display import display,FileLink
archive=ROOT/'V7_3_Context_Exploratory_1301.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT.iterdir():
        if p.is_file() and p.suffix.lower() in ('.csv','.jpg','.json'):
            z.write(p,p.name)
manifest={
    'model':QWEN_MODEL_ID,'min_tiled_boxes':MIN_TILED_BOXES,'n_sample':N_SAMPLE,
    'min_local_yes':MIN_LOCAL_YES,'context_multiplier':CONTEXT_MULTIPLIER,
    'require_original_full_yes':REQUIRE_ORIGINAL_FULL_YES,
    'base_predictions_sha256':hashlib.sha256((CACHEROOT/'v72_frozen_1301_predictions.csv').read_bytes()).hexdigest(),
    'description':'Exploratory post-hoc test on reused FSC-147 TEST; do not claim independent generalization.'}
(OUTPUT/'v73_protocol.json').write_text(json.dumps(manifest,indent=2),encoding='utf8')
with zipfile.ZipFile(archive,'a',zipfile.ZIP_DEFLATED) as z:z.write(OUTPUT/'v73_protocol.json','v73_protocol.json')
print('Đã tạo:',archive,'size:',archive.stat().st_size)
display(FileLink(str(archive)))